# Kaggle GPU pipeline: embeddings, clustering, and presentation figures

Runs the full trend-detection pipeline (embed -> static HDBSCAN baseline ->
dynamic micro-clustering with time decay) on a Kaggle **T4 GPU** notebook,
and saves both the computed embeddings and a set of presentation-ready PNG
charts.

**How to run this on Kaggle**

1. Create a Kaggle Dataset containing this repo's `data/master_dataset.csv`
   and the `src/` folder (so the notebook can import the same code the
   local project uses -- no logic is duplicated here).
2. Create a new Notebook, attach that dataset, and under *Settings* turn on
   **Accelerator: GPU T4 x2** (one T4 is enough) and **Internet: on** (the
   embedding model downloads from Hugging Face on first use).
3. Run all cells top to bottom.
4. Download `embeddings.npy`, `embeddings_ids.npy`, `embeddings_meta.json`
   from the notebook's Output/`/kaggle/working` and drop them into this
   repo's local `data/` folder -- the local dashboard (`embed_corpus_cached`)
   will detect and reuse them instead of re-embedding on CPU.
5. Download the `figures/` folder for slides.

**What this notebook is *not*:** Kaggle notebooks have no public ports, so
there is no reliable way to host the Streamlit dashboard itself from here.
This notebook is the GPU **compute + figures** step; the clickable demo
still runs locally with `streamlit run src/dashboard/app.py` (see the
project README).

## 0. Configuration

Leave everything below as-is for a normal run. `MAX_POSTS` caps the corpus
size (set to `None` to use every post); `HALF_LIFE_HOURS` controls how fast
the dynamic engine "forgets" inactive topics -- the default (6h) used for
the compressed synthetic demo is too short for a real, months-long
historical corpus, so this notebook uses a longer half-life.

`TEXT_MODE="title_text"` embeds `title + text` instead of bare `text` --
this matters most for YouTube, where `text` is just the comment body and
`title` is the video it was posted under. `SWEEP_ON=True` runs a small
grid over (similarity_threshold, half_life_hours, text_mode) scored on both
cluster quality and burst-detection performance, and overrides
`SIMILARITY_THRESHOLD` / `HALF_LIFE_HOURS` / `TEXT_MODE` above with the
winner -- set it `False` to skip straight to a single run with the fixed
values (much faster re-runs once you've already found good settings).


In [ ]:
# Set these manually only if auto-detection in the next section fails.
MANUAL_PROJECT_ROOT = None  # e.g. Path("/kaggle/input/cse-445-project")
MANUAL_DATA_PATH = None  # e.g. Path("/kaggle/input/cse-445-project/data/master_dataset.csv")

MAX_POSTS = None  # e.g. 20000 to cap the run; None = use the full corpus
HALF_LIFE_HOURS = 36
SIMILARITY_THRESHOLD = 0.55
TEXT_MODE = "title_text"  # "text" or "title_text" -- see markdown above

SWEEP_ON = True  # grid-search similarity/half-life/text_mode, then use the winner
SWEEP_MAX_POSTS = 4000  # subset size for the sweep itself (kept small - it multiplies out)

N_BURSTS = 6           # injected burst topics for the detection benchmark (max 6 available)
POSTS_PER_BURST = 80
BURST_SOURCE_PLATFORM = "reddit"  # which real platform's posts serve as burst-injection background


## 1. Install the packages Kaggle's base image doesn't already have

`torch` ships preinstalled (CUDA-enabled) on Kaggle GPU images, so it is
deliberately *not* reinstalled here -- doing so risks replacing it with a
CPU-only build.

In [ ]:
import subprocess
import sys


def _pip_install(*packages: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)


for module_name, pip_name in [
    ("sentence_transformers", "sentence-transformers"),
    ("hdbscan", "hdbscan"),
    ("umap", "umap-learn"),
]:
    try:
        __import__(module_name)
    except ImportError:
        print(f"Installing {pip_name} ...")
        _pip_install(pip_name)

print("Dependencies ready.")

## 2. Locate the project code and dataset

Searches the attached Kaggle dataset(s) under `/kaggle/input` for the
`src/ml_engine` package and `data/master_dataset.csv`, wherever they landed
after upload/zipping. Falls back to running in-place if this notebook is
opened locally inside the repo's own `notebooks/` folder.

In [ ]:
from pathlib import Path


def _find_project_root(search_roots: list[Path]) -> Path | None:
    for root in search_roots:
        if not root.exists():
            continue
        for candidate in root.rglob("ml_engine"):
            if (candidate / "vectorizer.py").exists():
                return candidate.parent.parent  # .../src/ml_engine -> project root
    return None


def _find_master_csv(search_roots: list[Path]) -> Path | None:
    for root in search_roots:
        if not root.exists():
            continue
        matches = list(root.rglob("master_dataset.csv"))
        if matches:
            return matches[0]
    return None


if MANUAL_PROJECT_ROOT is not None:
    PROJECT_ROOT = Path(MANUAL_PROJECT_ROOT)
else:
    PROJECT_ROOT = _find_project_root([Path("/kaggle/input"), Path.cwd(), Path.cwd().parent])
    if PROJECT_ROOT is None and (Path.cwd().parent / "src").exists():
        PROJECT_ROOT = Path.cwd().parent  # running locally from notebooks/

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the 'src' package. Set MANUAL_PROJECT_ROOT in the "
        "Configuration cell to the folder containing 'src/ml_engine/vectorizer.py'."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if MANUAL_DATA_PATH is not None:
    DATA_PATH = Path(MANUAL_DATA_PATH)
else:
    DATA_PATH = _find_master_csv([Path("/kaggle/input"), PROJECT_ROOT / "data"])

if DATA_PATH is None or not DATA_PATH.exists():
    raise FileNotFoundError(
        "Could not find 'master_dataset.csv'. Set MANUAL_DATA_PATH in the "
        "Configuration cell, or make sure it was included in the attached "
        "Kaggle dataset."
    )

IS_KAGGLE = Path("/kaggle/working").exists()
OUTPUT_DIR = Path("/kaggle/working") if IS_KAGGLE else (PROJECT_ROOT / "data")
FIGURES_DIR = OUTPUT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)
print("Output dir:", OUTPUT_DIR)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.artifacts import corpus_fingerprint, save_bundle
from src.data_ingestion.timeline import platform_windows
from src.evaluation.burst_benchmark import evaluate_burst_detection
from src.evaluation.sliding_window_baseline import evaluate_burst_detection_sliding_window
from src.evaluation.injection import inject_bursts
from src.evaluation.metrics import score_dynamic, score_static
from src.evaluation.report import build_evaluation_report, save_evaluation_csvs
from src.evaluation.sweeps import best_params, sweep_dynamic_engine
from src.ml_engine.dynamic_engine import DynamicClusteringEngine, run_replay
from src.ml_engine.static_clustering import cluster_hdbscan, cluster_summary, reduce_umap
from src.ml_engine.vectorizer import TextVectorizer, embed_corpus_cached, load_corpus
from src.summarization.trend_labeler import heuristic_label

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected -- this will fall back to CPU (fine for testing, slow for 40k+ posts).")


## 3. Load the corpus

In [ ]:
df = load_corpus(DATA_PATH)

if MAX_POSTS is not None and len(df) > MAX_POSTS:
    df = (
        df.assign(_ts=pd.to_datetime(df["timestamp"], utc=True, format="ISO8601"))
        .sort_values("_ts")
        .tail(MAX_POSTS)
        .drop(columns="_ts")
        .reset_index(drop=True)
    )

CORPUS_FINGERPRINT = corpus_fingerprint(df)

print(f"Loaded {len(df)} posts (fingerprint {CORPUS_FINGERPRINT})")
print()
print(df["platform"].value_counts())
print()
windows = platform_windows(df)
print("Real per-platform time windows (note: these are typically disjoint - see README):")
windows


## 4. Figure: platform breakdown

Evidence that the corpus is genuinely multi-platform, not just multi-topic.

In [ ]:
platform_counts = df["platform"].value_counts()

fig, ax = plt.subplots(figsize=(7, 5))
platform_counts.plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_title("Posts per platform")
ax.set_xlabel("Platform")
ax.set_ylabel("Post count")
for i, value in enumerate(platform_counts.values):
    ax.text(i, value, f"{value:,}", ha="center", va="bottom")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_platform_breakdown.png", dpi=150)
plt.show()


## 5. Figure: posts over time, by platform

Each platform's historical dataset was collected over a different window, so this typically shows distinct time blocks per platform rather than a continuous overlapping stream - worth calling out during a presentation. See the per-platform window table above for the exact numbers.

In [ ]:
daily = df.assign(_date=pd.to_datetime(df["timestamp"], utc=True, format="ISO8601").dt.floor("D"))
daily_counts = daily.groupby(["_date", "platform"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(11, 5))
for platform in daily_counts.columns:
    ax.plot(daily_counts.index, daily_counts[platform], marker=".", markersize=3, label=platform)
ax.set_title("Daily post volume by platform")
ax.set_xlabel("Date")
ax.set_ylabel("Posts per day")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_posts_over_time.png", dpi=150)
plt.show()


## 6. Embed every post on the GPU (cached to disk)

`TextVectorizer` auto-detects CUDA, so this uses the T4 automatically. `embed_corpus_cached` skips re-embedding on a re-run of this notebook (or later, back in the local dashboard) as long as the same `post_id`s, model, and `text_mode` are used - see `src/ml_engine/vectorizer.py`.

In [ ]:
import time

EMBEDDINGS_PATH = OUTPUT_DIR / "embeddings.npy"
EMBEDDINGS_IDS_PATH = OUTPUT_DIR / "embeddings_ids.npy"
EMBEDDINGS_META_PATH = OUTPUT_DIR / "embeddings_meta.json"

vectorizer = TextVectorizer()  # device="cuda" automatically when available

t0 = time.time()
embeddings = embed_corpus_cached(
    df,
    vectorizer=vectorizer,
    text_mode=TEXT_MODE,
    embeddings_path=EMBEDDINGS_PATH,
    ids_path=EMBEDDINGS_IDS_PATH,
    meta_path=EMBEDDINGS_META_PATH,
)
print(f"Embedded {embeddings.shape[0]} posts (text_mode={TEXT_MODE!r}) into shape {embeddings.shape} in {time.time() - t0:.1f}s")
print("Saved to:", EMBEDDINGS_PATH)


## 7. Parameter sweep

Grids over `(similarity_threshold, half_life_hours, text_mode)`, scoring each cell on both cluster quality (ARI/NMI vs. the `source` pseudo-labels) and burst-detection performance (via a fresh semi-synthetic injection per cell - see `src/evaluation/sweeps.py`). Runs on a `SWEEP_MAX_POSTS`-sized subset since it multiplies out fast; the winner becomes this run's `SIMILARITY_THRESHOLD` / `HALF_LIFE_HOURS` / `TEXT_MODE`.

In [ ]:
if SWEEP_ON:
    sweep_df = df
    if len(sweep_df) > SWEEP_MAX_POSTS:
        sweep_df = (
            sweep_df.assign(_ts=pd.to_datetime(sweep_df["timestamp"], utc=True, format="ISO8601"))
            .sort_values("_ts")
            .tail(SWEEP_MAX_POSTS)
            .drop(columns="_ts")
            .reset_index(drop=True)
        )

    sweep_results = sweep_dynamic_engine(sweep_df)
    chosen = best_params(sweep_results)

    SIMILARITY_THRESHOLD = chosen["similarity_threshold"]
    HALF_LIFE_HOURS = chosen["half_life_seconds"] / 3600.0
    TEXT_MODE = chosen["text_mode"]

    print("Sweep winner:", chosen)
    with pd.option_context("display.max_colwidth", 30, "display.width", 160):
        display(sweep_results.head(10))

    sweep_results.to_csv(OUTPUT_DIR / "sweep_results.csv", index=False)

    fig, ax = plt.subplots(figsize=(8, 5))
    pivot = sweep_results.pivot_table(
        index="half_life_hours", columns="similarity_threshold", values="combined_score", aggfunc="mean"
    )
    im = ax.imshow(pivot.values, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("similarity_threshold")
    ax.set_ylabel("half_life_hours")
    ax.set_title("Sweep: combined score (mean over text_mode)")
    fig.colorbar(im, ax=ax, label="combined score")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "08_sweep_heatmap.png", dpi=150)
    plt.show()
else:
    sweep_results = pd.DataFrame()
    print("SWEEP_ON is False - using the fixed config values as-is.")

print(f"Using: similarity_threshold={SIMILARITY_THRESHOLD}, half_life_hours={HALF_LIFE_HOURS}, text_mode={TEXT_MODE!r}")


## 8. Static baseline: UMAP + HDBSCAN, with cluster-quality evaluation

A one-shot snapshot clustering, same as `src/ml_engine/static_clustering.py`. Scored against the `source` column (subreddit / YouTube topic / synthetic topic label) as a free, imperfect pseudo-label ground truth - `src/evaluation/metrics.py`.

In [ ]:
reduced = reduce_umap(embeddings, n_components=5)
static_labels_array = cluster_hdbscan(reduced, min_cluster_size=5, min_samples=5)

n_clusters = len(set(static_labels_array)) - (1 if -1 in static_labels_array else 0)
noise_ratio = float(np.mean(static_labels_array == -1))
print(f"HDBSCAN found {n_clusters} clusters ({noise_ratio:.1%} noise) across {len(df)} posts")

static_summary = cluster_summary(df, static_labels_array, text_column="title", samples_per_cluster=3)
static_eval = score_static(df, static_labels_array, embeddings=embeddings)
print("Static cluster-quality vs. source pseudo-labels:", static_eval)

static_labels_df = pd.DataFrame({"post_id": df["post_id"].values, "cluster": static_labels_array})
static_summary.head(10)


## 9. Figure: UMAP scatter of static clusters

In [ ]:
embeddings_2d = reduce_umap(embeddings, n_components=2)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    embeddings_2d[:, 0], embeddings_2d[:, 1], c=static_labels_array, cmap="tab20", s=8, alpha=0.7
)
ax.set_title(f"Static HDBSCAN clusters ({n_clusters} clusters, UMAP 2D projection)")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
fig.colorbar(scatter, ax=ax, label="cluster")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "03_umap_scatter.png", dpi=150)
plt.show()


## 10. Figure: cluster size histogram (HDBSCAN)

In [ ]:
cluster_sizes = (
    pd.Series(static_labels_array)[pd.Series(static_labels_array) != -1]
    .value_counts()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(cluster_sizes.values, bins=30, color="#55A868")
ax.set_title("Distribution of HDBSCAN cluster sizes (excluding noise)")
ax.set_xlabel("Cluster size (posts)")
ax.set_ylabel("Number of clusters")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "04_cluster_size_histogram.png", dpi=150)
plt.show()


## 11. Dynamic engine: online micro-clustering with time decay

Replays the corpus chronologically through the same `DynamicClusteringEngine` the dashboard uses (`src/ml_engine/dynamic_engine.py`), with `track_state_log=True` so state transitions (needed by the burst benchmark below) are recorded. Scored against the same `source` pseudo-labels as the static baseline, via per-post cluster assignments.

Labels use the offline heuristic labeler only (`heuristic_label`), not Gemini - no API key is needed or used on Kaggle.

In [ ]:
engine = DynamicClusteringEngine(
    similarity_threshold=SIMILARITY_THRESHOLD,
    half_life_seconds=HALF_LIFE_HOURS * 3600.0,
    track_state_log=True,
)

t0 = time.time()
replay = run_replay(
    df,
    vectorizer=vectorizer,
    engine=engine,
    text_mode=TEXT_MODE,
    embeddings_path=EMBEDDINGS_PATH,
    ids_path=EMBEDDINGS_IDS_PATH,
    meta_path=EMBEDDINGS_META_PATH,
)
dynamic_summary = replay.summary
print(f"Replay finished in {time.time() - t0:.1f}s")
print(f"{len(engine.registry.clusters)} clusters active at the end of the replay")
print()
print(dynamic_summary["state"].value_counts())

dynamic_eval = score_dynamic(replay.assignments, df)
print("Dynamic cluster-quality vs. source pseudo-labels:", dynamic_eval)

labels = {
    cluster_id: heuristic_label(list(cluster.recent_titles))
    for cluster_id, cluster in engine.registry.clusters.items()
    if cluster_id in set(dynamic_summary["cluster_id"])
}
dynamic_summary_labeled = dynamic_summary.copy()
dynamic_summary_labeled["label"] = dynamic_summary_labeled["cluster_id"].map(
    lambda cid: labels.get(cid, {}).get("label", "")
)
dynamic_summary_labeled["description"] = dynamic_summary_labeled["cluster_id"].map(
    lambda cid: labels.get(cid, {}).get("description", "")
)
dynamic_summary_labeled.head(10)


## 12. Figure: top dynamic clusters by weight

In [ ]:
top_clusters = dynamic_summary_labeled.sort_values("weight", ascending=False).head(15)
tick_labels = [
    f"#{cid} {labels.get(cid, {}).get('label', '')}"[:35] for cid in top_clusters["cluster_id"]
]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(tick_labels[::-1], top_clusters["weight"].values[::-1], color="#C44E52")
ax.set_title("Top 15 active clusters by decayed weight")
ax.set_xlabel("Weight")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "05_top_clusters_by_weight.png", dpi=150)
plt.show()


## 13. Figure: trending-state mix

In [ ]:
state_order = ["trending", "rising", "declining", "dormant"]
state_colors = {"trending": "#C44E52", "rising": "#DD8452", "declining": "#4C72B0", "dormant": "#8C8C8C"}
state_counts = dynamic_summary["state"].value_counts().reindex(state_order, fill_value=0)

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(state_counts.index, state_counts.values, color=[state_colors[s] for s in state_counts.index])
ax.set_title("Active clusters by trending state")
ax.set_xlabel("State")
ax.set_ylabel("Number of clusters")
for i, value in enumerate(state_counts.values):
    ax.text(i, value, str(value), ha="center", va="bottom")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "06_state_mix.png", dpi=150)
plt.show()


## 14. Figure: weight-over-time for the top clusters

Shows the time-decay + growth dynamics behind the growth-rate scores for the 6 heaviest clusters - a good "here's what the online algorithm is actually doing" slide.

In [ ]:
top6_ids = dynamic_summary.sort_values("weight", ascending=False)["cluster_id"].head(6)

fig, ax = plt.subplots(figsize=(10, 6))
for cluster_id in top6_ids:
    history = engine.velocity.history_df(cluster_id)
    if history.empty:
        continue
    label = f"#{cluster_id} {labels.get(cluster_id, {}).get('label', '')}"[:40]
    ax.plot(history["timestamp"], history["weight"], marker=".", markersize=4, label=label)

ax.set_title("Weight over time: top 6 clusters")
ax.set_xlabel("Time")
ax.set_ylabel("Decayed weight")
ax.legend(fontsize=8)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "07_velocity_top_clusters.png", dpi=150)
plt.show()


## 15. Burst-detection benchmark (semi-synthetic injection)

The primary evidence that the dynamic engine actually detects emerging trends, with a number attached instead of an unverified claim: paraphrased burst topics are injected into a real background corpus at known timestamps (`src/evaluation/injection.py`), then scored on detection latency, precision, and recall (`src/evaluation/burst_benchmark.py`).

In [ ]:
background = df[df["platform"] == BURST_SOURCE_PLATFORM].copy()
if background.empty:
    print(f"No {BURST_SOURCE_PLATFORM!r} posts in this corpus; falling back to the full corpus as background.")
    background = df.copy()

injected_df, burst_truth = inject_bursts(
    background, n_bursts=N_BURSTS, posts_per_burst=POSTS_PER_BURST, seed=42
)

burst_metrics = evaluate_burst_detection(
    injected_df,
    burst_truth,
    engine_params={
        "similarity_threshold": SIMILARITY_THRESHOLD,
        "half_life_seconds": HALF_LIFE_HOURS * 3600.0,
    },
    vectorizer=vectorizer,
    text_mode=TEXT_MODE,
    embeddings_path=OUTPUT_DIR / "burst_embeddings.npy",
    ids_path=OUTPUT_DIR / "burst_embeddings_ids.npy",
    meta_path=OUTPUT_DIR / "burst_embeddings_meta.json",
)

print(
    f"detection_rate={burst_metrics['detection_rate']:.2f} "
    f"precision={burst_metrics['precision']:.2f} recall={burst_metrics['recall']:.2f} "
    f"f1={burst_metrics['f1']:.2f}"
)
if burst_metrics["median_latency_seconds"] is not None:
    print(f"median latency: {burst_metrics['median_latency_seconds'] / 3600:.1f}h")

with pd.option_context("display.max_colwidth", 40):
    display(burst_metrics["per_burst"])

fig, ax = plt.subplots(figsize=(8, 5))
detected = burst_metrics["per_burst"]
labels_x = detected["topic_label"]
latencies_h = detected["latency_seconds"].apply(lambda s: s / 3600 if pd.notna(s) else None)
colors = ["#55A868" if d else "#C44E52" for d in detected["detected"]]
ax.bar(labels_x, latencies_h.fillna(0), color=colors)
ax.set_title("Burst-detection latency (green=detected, red=missed)")
ax.set_ylabel("Latency (hours)")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "09_burst_latency.png", dpi=150)
plt.show()


## 16. Sliding-window HDBSCAN baseline (the periodic-recluster comparison)

The natural "brute force" alternative to online micro-clustering: periodically re-cluster a trailing window of recent posts with UMAP + HDBSCAN from scratch (`src/evaluation/sliding_window_baseline.py`), track cluster continuity by nearest-centroid similarity across refreshes, and flag a cluster "surging" when it's new-and-large or growing fast. Scored on the exact same injected corpus as the dynamic engine above - this is the direct "is online actually better than just re-running HDBSCAN on a rolling window" comparison, on both detection latency *and* compute cost (posts reclustered).

In [ ]:
sliding_window_metrics = evaluate_burst_detection_sliding_window(
    injected_df,
    burst_truth,
    window_hours=max(24.0, HALF_LIFE_HOURS),
    refresh_interval_hours=HALF_LIFE_HOURS / 6.0,
    vectorizer=vectorizer,
    text_mode=TEXT_MODE,
    embeddings_path=OUTPUT_DIR / "sliding_window_embeddings.npy",
    ids_path=OUTPUT_DIR / "sliding_window_embeddings_ids.npy",
    meta_path=OUTPUT_DIR / "sliding_window_embeddings_meta.json",
)

comparison = pd.DataFrame(
    [
        {
            "method": "dynamic (online)",
            "detection_rate": burst_metrics["detection_rate"],
            "precision": burst_metrics["precision"],
            "median_latency_hours": (
                burst_metrics["median_latency_seconds"] / 3600
                if burst_metrics["median_latency_seconds"] is not None
                else None
            ),
            "posts_processed": len(injected_df),
        },
        {
            "method": "sliding-window HDBSCAN",
            "detection_rate": sliding_window_metrics["detection_rate"],
            "precision": sliding_window_metrics["precision"],
            "median_latency_hours": (
                sliding_window_metrics["median_latency_seconds"] / 3600
                if sliding_window_metrics["median_latency_seconds"] is not None
                else None
            ),
            "posts_processed": sliding_window_metrics["total_posts_reclustered"],
        },
    ]
)
print(comparison.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(comparison["method"], comparison["median_latency_hours"], color=["#4C72B0", "#DD8452"])
axes[0].set_title("Median detection latency")
axes[0].set_ylabel("hours")
plt.setp(axes[0].get_xticklabels(), rotation=15, ha="right")

axes[1].bar(comparison["method"], comparison["posts_processed"], color=["#4C72B0", "#DD8452"])
axes[1].set_title("Compute cost (posts processed)")
axes[1].set_ylabel("posts")
plt.setp(axes[1].get_xticklabels(), rotation=15, ha="right")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "10_dynamic_vs_sliding_window.png", dpi=150)
plt.show()


## 17. Per-platform evaluation

Static and dynamic cluster quality scored separately per platform - the honest view, given the platforms don't share a time window (see the window table above) so a single pooled score can hide platform-specific behavior.

In [ ]:
per_platform_rows = []
for platform, group in df.groupby("platform"):
    if len(group) < 10:
        continue
    group_labels = static_labels_df.set_index("post_id").loc[group["post_id"], "cluster"].values
    static_row = score_static(group, group_labels)
    dynamic_row = score_dynamic(replay.assignments, group)
    per_platform_rows.append({"platform": platform, "method": "static", **static_row})
    per_platform_rows.append({"platform": platform, "method": "dynamic", **dynamic_row})

per_platform_eval = pd.DataFrame(per_platform_rows)
per_platform_eval


## 18. Export artifacts

Writes everything into `OUTPUT_DIR/artifacts/` as one bundle (`src/artifacts.py`) and zips it - download `artifacts.zip` from this notebook's **Output** tab (backed by `/kaggle/working`) and unzip it into this repo's local `data/artifacts/`. The local Streamlit dashboard (`streamlit run src/dashboard/app.py`) then reads it directly - no local GPU, no re-embedding, no re-running clustering.

Kaggle notebooks have no public ports, so there is no reliable way to host the Streamlit dashboard itself from here - this notebook is the GPU **compute + evaluation + figures** step; the clickable demo still runs locally.

In [ ]:
evaluation_report = build_evaluation_report(
    static_metrics=static_eval,
    dynamic_metrics=dynamic_eval,
    burst_metrics=burst_metrics,
    sliding_window_metrics=sliding_window_metrics,
    sweep_results=sweep_results,
)

ARTIFACTS_DIR = OUTPUT_DIR / "artifacts"
save_bundle(
    ARTIFACTS_DIR,
    run_meta={
        "n_posts": len(df),
        "corpus_fingerprint": CORPUS_FINGERPRINT,
        "similarity_threshold": SIMILARITY_THRESHOLD,
        "half_life_hours": HALF_LIFE_HOURS,
        "text_mode": TEXT_MODE,
        "platform_counts": df["platform"].value_counts().to_dict(),
    },
    embeddings=embeddings,
    embeddings_ids=df["post_id"].astype(str).tolist(),
    embeddings_model_name=vectorizer.model_name,
    embeddings_text_mode=TEXT_MODE,
    static_labels=static_labels_df,
    static_umap2d=embeddings_2d,
    static_summary=static_summary,
    dynamic_assignments=replay.assignments,
    dynamic_history=replay.history,
    dynamic_summary=dynamic_summary_labeled,
    evaluation=evaluation_report,
)
save_evaluation_csvs(
    ARTIFACTS_DIR,
    burst_metrics=burst_metrics,
    sliding_window_metrics=sliding_window_metrics,
    sweep_results=sweep_results,
    per_platform=per_platform_eval,
)

import shutil
shutil.copytree(FIGURES_DIR, ARTIFACTS_DIR / "figures", dirs_exist_ok=True)

zip_path = OUTPUT_DIR / "artifacts"
shutil.make_archive(str(zip_path), "zip", root_dir=str(ARTIFACTS_DIR))
print("Artifact bundle:", ARTIFACTS_DIR)
print("Zipped to:", zip_path.with_suffix(".zip"))
print()
for path in sorted(ARTIFACTS_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(ARTIFACTS_DIR)}  ({path.stat().st_size / 1024:,.1f} KB)")
